# Recipe 7: Text as Data: Bag of Words, TF-IDF, and Embeddings
**EDPY 506 · Machine Learning: Theory and Applications · Recipe notebooks**

<a href="https://colab.research.google.com/github/okanbulut/edpy506/blob/main/recipes/notebooks/Recipe07_Text_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Models need numbers, so text must be turned into numbers first. **Bag of words** counts how often each word appears. **TF-IDF** turns those counts into weights that favour words that are frequent in *this* message but rare overall. **Embeddings** squeeze thousands of word columns into a few hundred dense numbers, so that words used in similar ways end up close together.

**Lectures:** 9 (NLP foundations)
**Data:** SMS Spam Collection (Almeida & Hidalgo, 2011): 5,572 real text messages labelled spam or ham (not spam), the same data as the Lecture 9 hands-on. It loads straight from GitHub. The recipe is identical for educational text such as forum posts, open-ended answers, or course reviews.
**Runtime:** about one minute on a regular (CPU) runtime. No setup or uploads are needed: just run the cells from top to bottom (**Runtime → Run all**).

**How this notebook is organized.** It follows the seven steps of the *ML Workflow Recipe* paper:
1. Define the problem · 2. Split the data · 3. Preprocess · 4. Baseline · 5. Train and tune · 6. Evaluate · 7. Interpret

After each output there is a **What you should see** note. Your numbers should match closely if you keep the default parameters. If you change the parameters, the numbers will change, and that is the point: try it and see what happens.

In [ ]:
# Run this cell first. It loads the tools we use.
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 110

## ✏️ Change these parameters
This is the **only cell you need to edit**. Change a value, then choose **Runtime → Run all** to rerun the whole recipe.

In [ ]:
# ===== CHANGE THESE PARAMETERS =====
TEST_SIZE = 0.2          # share of messages kept aside for the final test
SEED = 506               # keeps the random split the same every run
USE_TFIDF = True         # True = TF-IDF weights, False = plain word counts (bag of words)
NGRAM_MAX = 1            # 1 = single words; 2 = also pairs of words ("free entry")
MIN_DF = 2               # ignore words that appear in fewer than this many messages
N_DIMENSIONS = 100       # size of the embeddings we build in Step 7 (try 20, 100, 300)
QUERY_WORD = "prize"      # find the words closest to this one in embedding space
MY_MESSAGE = "Congratulations! You have won a free prize. Call now to claim it."   # classify your own text
# ====================================

## Load the data

In [ ]:
import pandas as pd
URLS = ["https://raw.githubusercontent.com/okanbulut/edpy506/main/data/sms_spam.tsv",            # course copy
        "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"]  # backup copy
for url in URLS:
    try:
        sms = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])
        break
    except Exception:
        print("Could not reach", url, "- trying the backup copy.")
sms["spam"] = (sms["label"] == "spam").astype(int)
print(f"{len(sms):,} messages; {sms['spam'].mean():.1%} are spam.")

## Step 1. Define the problem
**Question:** can we tell spam from ordinary messages using only the words in them?
This is classification with text as the input. Look at the class balance first: it sets the baseline.

In [ ]:
print(sms["label"].value_counts())
sms.groupby("label").head(3)

> **What you should see:** 4,825 ham and 747 spam messages: 13.4% spam. Spam is full of words like *free*, *win*, *prize* and phone numbers; ham messages are short and chatty.

## Step 2. Split the data
Split **before** building the vocabulary, so the test messages cannot influence which words the model knows.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(sms["text"], sms["spam"], test_size=TEST_SIZE,
                                                    random_state=SEED, stratify=sms["spam"])
print(f"Training: {len(X_train):,} messages   Test: {len(X_test):,} messages")

## Step 3. Preprocess: from words to numbers
**Tokenization** splits text into pieces (tokens). The vectorizer lowercases, splits on anything that is not a letter or digit, and builds one column per word.

In [ ]:
msg = X_train.iloc[0]
print("Message:   ", msg)
print("split():   ", msg.split()[:12])
print("Tokenized: ", re.findall(r"(?u)\b\w\w+\b", msg.lower())[:12])

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
Vec = TfidfVectorizer if USE_TFIDF else CountVectorizer
vectorizer = Vec(ngram_range=(1, NGRAM_MAX), min_df=MIN_DF)
Xtr = vectorizer.fit_transform(X_train)      # learn the vocabulary from training messages only
Xte = vectorizer.transform(X_test)
print(f"\n{Xtr.shape[1]:,} columns (one per word). {1 - Xtr.nnz / np.prod(Xtr.shape):.2%} of the table is zeros.")

> **What you should see:** `split()` keeps capital letters and punctuation attached; the tokenizer lowercases and keeps only words of two or more letters. With `MIN_DF = 2` the vocabulary has about 3,600 words, and 99.7% of the table is zeros, because each message uses only a handful of them.

## Step 4. Baseline
Predict "not spam" for every message.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
print(f"Baseline accuracy: {1 - y_test.mean():.1%}  (and it catches 0% of spam)")

> **What you should see:** Baseline accuracy of about 86.6%, but it catches 0% of spam. High accuracy can be useless when one class is rare.

## Step 5. Train a logistic regression on the word columns

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=2000, C=10)
clf.fit(Xtr, y_train)
print("Trained.")

## Step 6. Evaluate
Accuracy looks great because most messages are not spam. Look at **recall for spam**: how many spam messages did we catch?

In [ ]:
pred = clf.predict(Xte)
print(f"Accuracy: {accuracy_score(y_test, pred):.1%}\n")
print(classification_report(y_test, pred, target_names=["ham", "spam"], digits=2))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["ham", "spam"]).plot(cmap="Blues", colorbar=False)
plt.title("Confusion matrix (test messages)"); plt.show()

> **What you should see:** Accuracy of about 97.8%. The important number is **spam recall of about 0.86**: 128 of 149 spam messages are caught and 21 slip through. Only 3 ordinary messages are wrongly sent to spam (precision about 0.98).

## Step 7. Interpret
### Which words push a message toward spam?
Each word has one coefficient. Large positive = spam signal; large negative = ordinary-message signal.

In [ ]:
words = np.array(vectorizer.get_feature_names_out())
coef = clf.coef_[0]
top = pd.DataFrame({"spam words": words[np.argsort(coef)[::-1][:15]],
                    "ham words": words[np.argsort(coef)[:15]]})
display(top.T)

missed = X_test[(y_test == 1) & (pred == 0)]
print(f"{len(missed)} spam messages slipped through. A few of them:")
for m in missed.head(4): print("  -", m[:110])

p = clf.predict_proba(vectorizer.transform([MY_MESSAGE]))[0, 1]
print(f"\nYour message: \"{MY_MESSAGE}\"\nP(spam) = {p:.0%}")

> **What you should see:** Spam words include *txt*, *text*, *uk*, *call*, *150p*, *claim* and *free*; ham words include *me*, *my*, *ok*, *later* and *home*, plus *gt* and *lt* (leftover HTML codes). The missed spam messages are ones that sound personal or chatty. The sample message gets P(spam) of about 99%.

### From word counts to embeddings
Bag of words treats "prize" and "award" as unrelated columns. **Latent semantic analysis (LSA)** compresses the word table into `N_DIMENSIONS` dense numbers per message (it is PCA for text, Recipe 4). Words that appear in similar messages end up with similar numbers.
This is the counting-based cousin of word2vec and GloVe from Lecture 9, and it runs in seconds.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
svd = TruncatedSVD(n_components=N_DIMENSIONS, random_state=SEED)
E_train = svd.fit_transform(Xtr)                 # each message -> N_DIMENSIONS numbers
E_test = svd.transform(Xte)
print(f"Each message is now {E_train.shape[1]} numbers instead of {Xtr.shape[1]:,}. "
      f"They keep {svd.explained_variance_ratio_.sum():.0%} of the variance.")

# Word embeddings: one row per word
word_vecs = normalize(svd.components_.T)
vocab = {w: i for i, w in enumerate(words)}
if QUERY_WORD in vocab:
    sims = word_vecs @ word_vecs[vocab[QUERY_WORD]]
    nearest = [words[j] for j in np.argsort(sims)[::-1][1:11]]
    print(f"Words closest to '{QUERY_WORD}':", ", ".join(nearest))
else:
    print(f"'{QUERY_WORD}' is not in the vocabulary. Try another word.")

emb_clf = LogisticRegression(max_iter=2000, C=10).fit(E_train, y_train)
emb_pred = emb_clf.predict(E_test)
print(f"\nLogistic regression on {N_DIMENSIONS}-number embeddings: accuracy {accuracy_score(y_test, emb_pred):.1%}, "
      f"spam recall {((emb_pred == 1) & (y_test == 1)).sum() / y_test.sum():.0%}")

> **What you should see:** Each message shrinks from about 3,600 numbers to 100, keeping about 32% of the variance. The words closest to *prize* are *guaranteed*, *won*, *claim*, *2000*, *900* and *valid*: words that appear in the same messages. A classifier on just 100 numbers gets about 97.4% accuracy and 85% spam recall, almost as good as the full word table.

In [ ]:
# A 2-D map of the messages (first two embedding dimensions)
two = TruncatedSVD(n_components=2, random_state=SEED).fit_transform(Xtr)
for lab, col, mk in ((0, "#70C2EF", "o"), (1, "#E07B39", "^")):
    m = (y_train == lab).values
    plt.scatter(two[m, 0], two[m, 1], s=8, alpha=0.5, color=col, marker=mk, label=["ham", "spam"][lab])
plt.xlabel("dimension 1"); plt.ylabel("dimension 2"); plt.legend(); plt.title("Messages in a 2-D embedding space")
plt.show()

> **What you should see:** Spam (orange triangles) bunches together in one region; ham (blue dots) spreads out widely. Even two dimensions separate a lot of the spam, though the groups overlap.

## Try this next
1. Set `USE_TFIDF = False` (plain counts). Does accuracy change? Does spam recall?
2. Set `NGRAM_MAX = 2`. More columns, but is the model better? (In Lecture 9 it was slightly worse.)
3. Change `QUERY_WORD` to `"happy"`, `"home"`, or `"txt"`. Do the neighbours make sense?
4. Type your own `MY_MESSAGE`. Can you write a spam message the model misses?

---
## Where to go next
- **Companion paper:** *Companion papers/Recipe07_Text_Embeddings_Companion.pdf* explains the idea behind this recipe in plain words.
- **Workflow:** *01_ML_Workflow_Recipe.pdf* has a checklist for every step used here.
- **Which method?** *02_Which_Method_Flowchart.pdf* helps you pick the right recipe for a new question.
- **Next recipe:** Recipe 8 swaps the home-made embeddings for a pretrained transformer and asks a small language model to do the job.

*Data citation:* Almeida, T. A., Gómez Hidalgo, J. M., & Yamakami, A. (2011). Contributions to the study of SMS spam filtering. *Proceedings of the 11th ACM Symposium on Document Engineering*, 259–262.